# Data Preperation

Merges GSWA HyLogger Data with WAMEX data into one master dataset, cleanses and splits dataset.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
ROOT = Path.cwd().parent if Path.cwd().name == "ml_pipeline" else Path.cwd()

DATA_DIR = ROOT / "data"
CSV_DIR = DATA_DIR / "INTERIM_HyLogger_Dataset_Package_GDA2020_DB" / "DATABASES" / "CSV"
MINERAL_LOGS_DIR = DATA_DIR / "extracted_data" / "mineral_logs"
MANIFEST = DATA_DIR / "extracted_data" / "mineral_logs_manifest.csv"
RANDOM_STATE = 67

### Mineral logs + Au assays

Each mineral log row is matched to the `WAMEX_AssayFlatData` interval (same `CollarId`, resolved via `HyLogger_Dataset_Index`) that contains the  row's depth midpoint. Rows without an `Au_PPM` value are dropped. `Dataset_ID` is the first column.

In [7]:
import sys

sys.path.insert(0, str(ROOT / "src"))
from gswa_hackathon.build_dataset import build_dataset

pd.set_option("display.max_columns", None)

df = build_dataset()
print(df.shape)
df.head()

(7748, 57)


,Dataset_ID,CollarId,Depth_from_(m),Depth_to_(m),Assay_FromDepth,Assay_ToDepth,Au_PPM,Kaolin_(System_SWIR),White-Mica_(System_SWIR),Smectite_(System_SWIR),Other-AlOH_(System_SWIR),Chlorite_(System_SWIR),Dark-Mica_(System_SWIR),Amphibole_(System_SWIR),Serpentine_(System_SWIR),Other-MgOH_(System_SWIR),Epidote_(System_SWIR),Tourmaline_(System_SWIR),Carbonate_(System_SWIR),Sulphate_(System_SWIR),Invalid_(System_SWIR),VIS-SWIR_Albedo,900D_Fe,900W_Fe,1900D_Water,2200D_AlOH,2200W_AlOH,2250D_FeOH,2250W_FeOH,Misc-Silicate_(System_TIR),Silica_(System_TIR),K-Feldspar_(System_TIR),Plagioclase_(System_TIR),Garnet_(System_TIR),Pyroxene_(System_TIR),Olivine_(System_TIR),Zeolite_(System_TIR),Kaolin_(System_TIR),White-Mica_(System_TIR),Smectite_(System_TIR),Other-AlOH_(System_TIR),Chlorite_(System_TIR),Dark-Mica_(System_TIR),Amphibole_(System_TIR),Serpentine_(System_TIR),Other-MgOH_(System_TIR),Epidote_(System_TIR),Tourmaline_(System_TIR),Carbonate_(System_TIR),Sulphate_(System_TIR),Phosphate_(System_TIR),Oxide_(System_TIR),INVALID_(System_TIR),8625D_Quartz,11300D_Carbonate,11300W_Carbonate,Invalid_(System_TIR)
0,16CADD001,3991008,113.0,114.0,113.2,114.0,0.04,NaN,0.785757,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.047195,NaN,0.167048,0.227175,0.076205,912.56799,0.059636,0.088736,2191.0605,0.002204,2254.2426,NaN,0.760987,0.021578,NaN,NaN,0.0219,NaN,NaN,NaN,0.038157,NaN,NaN,NaN,NaN,0.026851,NaN,NaN,NaN,NaN,0.109909,0.020617,NaN,NaN,NaN,0.104198,NaN,11489.371,NaN
1,16CADD001,3991008,114.0,115.0,114.0,115.0,0.02,NaN,0.761201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.203490,NaN,0.035309,0.192610,0.047196,917.11669,0.016377,0.065511,2191.2216,0.002667,2254.5463,NaN,0.695542,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.097945,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.206513,NaN,NaN,NaN,NaN,0.156546,0.020309,11280.979,NaN
2,16CADD001,3991008,115.0,116.0,115.0,116.0,0.02,NaN,0.779617,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.191897,NaN,0.028486,0.220350,0.055675,909.49853,0.019801,0.067166,2190.5102,0.002305,2255.9343,NaN,0.684421,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.137218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.178361,NaN,NaN,NaN,NaN,0.147508,0.019458,11279.096,NaN
3,16CADD001,3991008,116.0,117.0,116.0,117.0,0.03,NaN,0.825653,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.168769,NaN,0.005578,0.262213,0.038088,919.94647,0.021303,0.093491,2190.3549,0.002075,2252.7368,NaN,0.655389,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.151639,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.189342,NaN,NaN,0.00363,NaN,0.140524,0.019169,11352.566,NaN
4,16CADD001,3991008,117.0,118.0,117.0,118.0,0.05,NaN,0.887133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.108579,NaN,0.004288,0.261393,0.061666,915.53759,0.022842,0.083664,2190.5119,0.002070,2252.5795,NaN,0.731636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.154853,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.113511,NaN,NaN,NaN,NaN,0.141961,0.019044,11397.328,NaN


### DF FOR SWIR

In [17]:
ID_COLUMNS = ["Dataset_ID", "Depth_from_(m)", "Depth_to_(m)", "Assay_FromDepth", "Assay_ToDepth"]

TARGET = "Au_PPM"

SPECTRAL_FEATURES_SWIR = ["900D_Fe", "900W_Fe", "1900D_Water", "2200D_AlOH", "2200W_AlOH", "2250D_FeOH", "2250W_FeOH"]

df_swir = df[ID_COLUMNS + [TARGET] + SPECTRAL_FEATURES_SWIR].copy()

df_swir.info(verbose=True)
display(df_swir.head())
display(df_swir.describe())

<class 'pandas.DataFrame'>
RangeIndex: 7748 entries, 0 to 7747
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Dataset_ID       7748 non-null   str    
 1   Depth_from_(m)   7748 non-null   float64
 2   Depth_to_(m)     7748 non-null   float64
 3   Assay_FromDepth  7748 non-null   float64
 4   Assay_ToDepth    7748 non-null   float64
 5   Au_PPM           7748 non-null   float64
 6   900D_Fe          6014 non-null   float64
 7   900W_Fe          5916 non-null   float64
 8   1900D_Water      7748 non-null   float64
 9   2200D_AlOH       7748 non-null   float64
 10  2200W_AlOH       7748 non-null   float64
 11  2250D_FeOH       7748 non-null   float64
 12  2250W_FeOH       7748 non-null   float64
dtypes: float64(12), str(1)
memory usage: 787.0 KB


,Dataset_ID,Depth_from_(m),Depth_to_(m),Assay_FromDepth,Assay_ToDepth,Au_PPM,900D_Fe,900W_Fe,1900D_Water,2200D_AlOH,2200W_AlOH,2250D_FeOH,2250W_FeOH
0,16CADD001,113.0,114.0,113.2,114.0,0.04,0.076205,912.56799,0.059636,0.088736,2191.0605,0.002204,2254.2426
1,16CADD001,114.0,115.0,114.0,115.0,0.02,0.047196,917.11669,0.016377,0.065511,2191.2216,0.002667,2254.5463
2,16CADD001,115.0,116.0,115.0,116.0,0.02,0.055675,909.49853,0.019801,0.067166,2190.5102,0.002305,2255.9343
3,16CADD001,116.0,117.0,116.0,117.0,0.03,0.038088,919.94647,0.021303,0.093491,2190.3549,0.002075,2252.7368
4,16CADD001,117.0,118.0,117.0,118.0,0.05,0.061666,915.53759,0.022842,0.083664,2190.5119,0.002070,2252.5795


,Depth_from_(m),Depth_to_(m),Assay_FromDepth,Assay_ToDepth,Au_PPM,900D_Fe,900W_Fe,1900D_Water,2200D_AlOH,2200W_AlOH,2250D_FeOH,2250W_FeOH
count,7748.000000,7748.000000,7748.000000,7748.000000,7748.000000,6014.000000,5916.000000,7748.000000,7748.000000,7748.000000,7748.000000,7748.000000
mean,420.940243,421.940243,420.694895,422.201155,0.317079,0.044047,907.038196,0.018523,0.062545,2197.486166,0.064908,2253.438833
std,352.626904,352.626904,352.715536,352.544173,5.105411,0.020451,18.183437,0.026699,0.086247,11.255512,0.046021,3.650031
min,0.000000,1.000000,0.000000,1.000000,-1.000000,0.003280,846.770320,0.001287,0.001661,2138.478200,0.001670,2237.044600
25%,153.000000,154.000000,152.000000,154.000000,0.003000,0.032656,896.329218,0.004957,0.009733,2191.236875,0.022774,2251.850325
50%,307.000000,308.000000,306.500000,308.000000,0.007000,0.039198,905.244865,0.009030,0.025064,2196.541000,0.059083,2253.249450
75%,620.250000,621.250000,620.000000,622.000000,0.100000,0.050252,915.523582,0.018805,0.078744,2204.148350,0.100597,2256.167150
max,1785.000000,1786.000000,1784.900000,1785.900000,438.000000,0.410704,992.036070,0.185014,0.524566,2236.118900,0.254166,2273.386200


### DF FOR TIR 

In [19]:
ID_COLUMNS = ["Dataset_ID", "Depth_from_(m)", "Depth_to_(m)", "Assay_FromDepth", "Assay_ToDepth"]

TARGET = "Au_PPM"

SPECTRAL_FEATURES_TIR = ["8625D_Quartz", "11300D_Carbonate", "11300W_Carbonate"]

df_tir = df[ID_COLUMNS + [TARGET] + SPECTRAL_FEATURES_TIR].copy()

df_tir.info(verbose=True)
display(df_tir.head())
display(df_tir.describe())

<class 'pandas.DataFrame'>
RangeIndex: 7748 entries, 0 to 7747
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Dataset_ID        7748 non-null   str    
 1   Depth_from_(m)    7748 non-null   float64
 2   Depth_to_(m)      7748 non-null   float64
 3   Assay_FromDepth   7748 non-null   float64
 4   Assay_ToDepth     7748 non-null   float64
 5   Au_PPM            7748 non-null   float64
 6   8625D_Quartz      7447 non-null   float64
 7   11300D_Carbonate  5977 non-null   float64
 8   11300W_Carbonate  7553 non-null   float64
dtypes: float64(8), str(1)
memory usage: 544.9 KB


,Dataset_ID,Depth_from_(m),Depth_to_(m),Assay_FromDepth,Assay_ToDepth,Au_PPM,8625D_Quartz,11300D_Carbonate,11300W_Carbonate
0,16CADD001,113.0,114.0,113.2,114.0,0.04,0.104198,NaN,11489.371
1,16CADD001,114.0,115.0,114.0,115.0,0.02,0.156546,0.020309,11280.979
2,16CADD001,115.0,116.0,115.0,116.0,0.02,0.147508,0.019458,11279.096
3,16CADD001,116.0,117.0,116.0,117.0,0.03,0.140524,0.019169,11352.566
4,16CADD001,117.0,118.0,117.0,118.0,0.05,0.141961,0.019044,11397.328


,Depth_from_(m),Depth_to_(m),Assay_FromDepth,Assay_ToDepth,Au_PPM,8625D_Quartz,11300D_Carbonate,11300W_Carbonate
count,7748.000000,7748.000000,7748.000000,7748.000000,7748.000000,7447.000000,5977.000000,7553.000000
mean,420.940243,421.940243,420.694895,422.201155,0.317079,0.133597,0.012815,11350.111721
std,352.626904,352.626904,352.715536,352.544173,5.105411,0.037411,0.011385,80.043661
min,0.000000,1.000000,0.000000,1.000000,-1.000000,0.001409,0.001002,11202.535000
25%,153.000000,154.000000,152.000000,154.000000,0.003000,0.119299,0.003261,11296.752000
50%,307.000000,308.000000,306.500000,308.000000,0.007000,0.136987,0.009800,11345.645000
75%,620.250000,621.250000,620.000000,622.000000,0.100000,0.157101,0.020429,11378.429000
max,1785.000000,1786.000000,1784.900000,1785.900000,438.000000,0.229609,0.172006,11696.444000
